### Imports

In [1]:
import json
from pathlib import Path

import optuna
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.config import CONFIG
from src.lstm_utils.lstm_dataset import LSTMDataset
from src.lstm_utils.lstm_tokeniser import LSTMTokeniser
from src.lstm_utils.lstm_training import train_one_epoch, validate
from src.models.lstm_classifier import LSTMClassifier
from src.utils import run_sweep, plot_training_history


/Users/jackrong/University/34812-cwk-S-Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
TRAIN_DATA_PATH = Path(f'../data/train.csv')
VAL_DATA_PATH = Path(f'../data/dev.csv')
BENCHMARK_DATA_PATH = Path(f'../data/NLI_trial.csv')
TEST_DATA_PATH = Path(f'../data/test.csv')

HYPERPARAMETERS_PATH = Path(f'../hyperparameters/lstm.json')
MODEL_PATH = Path(f'../models/lstm.pt')

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Set a constant seed for reproducability

In [4]:
generator = torch.manual_seed(CONFIG.seed)

### Verify tokenising works

In [5]:
def verify_tokeniser() -> None:
    train_df = pd.read_csv(TRAIN_DATA_PATH)
    sample = train_df.sample(1).iloc[0]
    sample_premise = sample['premise']

    lstm_tokeniser = LSTMTokeniser()
    premise_ids, premise_mask = lstm_tokeniser.encode(sample_premise, CONFIG.lstm.max_length)
    print(f'Sample premise: {sample_premise}')
    print(f'Premise ids: {premise_ids}')
    print(f'Premise mask: {premise_mask}')
    assert(len(premise_ids) == CONFIG.lstm.max_length)

# verify_tokeniser()

### Set up Datasets and DataLoaders

In [6]:
lstm_tokeniser = LSTMTokeniser()

train_dataset = LSTMDataset(TRAIN_DATA_PATH, lstm_tokeniser)
val_dataset = LSTMDataset(VAL_DATA_PATH, lstm_tokeniser)
benchmark_dataset = LSTMDataset(BENCHMARK_DATA_PATH, lstm_tokeniser)

train_dataloader = DataLoader(train_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)
benchmark_dataset = DataLoader(benchmark_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)

### Hyperparameter Tuning

In [8]:
if CONFIG.lstm.hyperparameter_tuning.should_run:
    run_sweep(HYPERPARAMETERS_PATH, CONFIG.lstm.hyperparameter_tuning.trials)

[I 2026-03-30 04:59:59,406] A new study created in memory with name: no-name-a71197a9-983b-4790-8214-5e895498ab97


Running hyperparameter sweep...


Validating: 100%|██████████| 27/27 [00:03<00:00,  7.78batches/s]
[I 2026-03-30 05:03:57,278] Trial 0 finished with value: 0.6655285035629454 and parameters: {'lr': 0.00014731757424990414}. Best is trial 0 with value: 0.6655285035629454.
Training:  49%|████▉     | 47/95 [00:46<00:47,  1.00batches/s]
[W 2026-03-30 05:07:23,811] Trial 1 failed with parameters: {'lr': 0.001449790061987001} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/jackrong/University/34812-cwk-S-Project/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/m2/zhj6zm9x5b93941_pktttl300000gn/T/ipykernel_11973/3178373575.py", line 10, in objective
    train_one_epoch(device, model, criterion, optimiser, train_dataloader)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jackrong/University/34812-cwk-S-Project/src/lstm_utils/lstm_training.py

KeyboardInterrupt: 

### Training Loop

In [ ]:
if __name__ == '__main__':
    hyperparameters = json.load(open(HYPERPARAMETERS_PATH, 'r'))
    print(f'Hyperparameters used: {hyperparameters}')
    print()

    model = LSTMClassifier(lstm_tokeniser).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['lr'])
    criterion = nn.CrossEntropyLoss()

    print(f'Training LSTM...')
    best_acc = 0.0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    for epoch in range(CONFIG.epochs):
        print(f'[Epoch {epoch + 1}/{CONFIG.epochs}]')

        train_loss, train_acc = train_one_epoch(device, model, criterion, optimizer, train_dataloader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate(device, model, criterion, val_dataloader)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)

        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_acc * 100:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_acc * 100:.2f}%')
        print()

    print(f'Best model had an accuracy of {best_acc * 100:.2f}%.')
    print(f'Running final test...')
    model.load_state_dict(torch.load(MODEL_PATH))

    # TODO: Benchmark
    plot_training_history(train_losses, train_accs, val_losses, val_accs)